In [25]:
# Cell 1: Imports and Configuration

import csv
import logging
import time
from datetime import datetime
from google_play_scraper import Sort, reviews
import schedule

In [26]:
# --- Configuration ---
# Define constants for the application ID and review fetching parameters.
# This makes it easier to modify these values without searching through the code.
APP_ID = 'com.boa.boaMobileBanking' # Removed leading space
LANG = 'en'
COUNTRY = 'us'
SORT_ORDER = Sort.NEWEST
REVIEW_COUNT = 5000
FILTER_SCORE = None # Set to an integer (1-5) to filter by star rating, or None for all scores.
CSV_FILENAME = 'BOA-MobileBanking_reviews.csv'
LOG_FILENAME = 'scraper.log'

In [27]:
# --- Logging Setup ---
# Configure logging to write messages to a file.
# The format includes timestamp, log level, and the message.
logging.basicConfig(
    filename=LOG_FILENAME,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__) # Get a logger instance for this module

# --- Core Scraping Function ---
def scrape_play_store_reviews():
    """
    Fetches reviews from the Google Play Store for Bank of Abyssinia Mobile Banking App
    and saves them to a CSV file.
    """
    logger.info(f"Starting review scraping for APP_ID: {APP_ID}")

    try:
        # Fetch reviews from Google Play.
        results, _ = reviews(
            APP_ID,
            lang=LANG,
            country=COUNTRY,
            sort=SORT_ORDER,
            count=REVIEW_COUNT,
            filter_score_with=FILTER_SCORE
        )
        logger.info(f"Successfully fetched {len(results)} reviews.")

        # Prepare the data for CSV writing
        if results:
            # Define CSV header based on the keys we want to save
            # You can customize these based on what 'results' actually contain
            # and what you need. This is a comprehensive list.
            csv_header = [
                'reviewId', 'userName', 'score', 'at', 'content', 'replyContent',
                'repliedAt', 'version', 'thumbsUpCount', 'appVersion', 'url'
            ]

            # Save to CSV
            with open(CSV_FILENAME, mode='w', encoding='utf-8', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=csv_header)
                writer.writeheader()
                for review_data in results:
                    # Create a dictionary for the row, handling missing keys
                    # This ensures all fields in csv_header are present,
                    # even if a review doesn't have a specific key.
                    row = {key: review_data.get(key, '') for key in csv_header}
                    writer.writerow(row)

            logger.info(f"Reviews saved to {CSV_FILENAME}")
        else:
            logger.warning("No reviews fetched to save.")

    except Exception as e:
        logger.error(f"Error occurred while scraping reviews: {e}", exc_info=True) # exc_info for traceback


In [28]:

# --- Scheduling Function (Optional, based on 'schedule' import) ---
def schedule_scraper():
    """
    Schedules the 'scrape_play_store_reviews' function to run daily.
    This function sets up the scheduler and enters a loop to keep it running.
    """
    logger.info("Setting up daily scraping schedule.")
    # Schedule the function to run at a specific time each day.
    # You can adjust '03:00' to your desired time.
    schedule.every().day.at("03:00").do(scrape_play_store_reviews)
    logger.info("Scraper scheduled to run daily at 03:00 AM.")

    while True:
        # Run pending scheduled tasks.
        schedule.run_pending()
        # Pause execution for a short period to avoid busy-waiting.
        time.sleep(1)

In [29]:

# --- Main Execution Block ---
# This ensures that the code inside this block only runs when the script is executed directly
# (not when imported as a module into another script).
if __name__ == "__main__":
    # You can choose to run the scraper once immediately, or schedule it.
    # To run once:
    scrape_play_store_reviews()

    # To schedule daily runs, uncomment the line below:
    # schedule_scraper()